# 4g. Segment cuts

**What an investor reads into this.** Company-wide averages can hide a segment that is failing or one that carries the business. A buyer wants to know where growth and churn are concentrated, because that shapes the plan after the deal (which channels to fund, which customer types to target) and where the risk sits. Segment results are only worth acting on when the segment is large enough; small groups swing on a handful of customers.

Measures (`docs/metric_definitions.md`): net revenue and growth by year; logo churn is the annual December-to-December measure for 2025 (not tenure-matched); NRR and GRR are for 2025. Segments with fewer than 30 customers active in December 2024 are flagged as too small. A segment is flagged as a risk if its 2025 logo churn is more than 5 points above the company rate or its NRR more than 5 points below; as a strength if the reverse. Section 3 adds a tenure-matched view (12-month retention for older against newer cohorts).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from nb_utils import connect, q, sql_file, save_table
import charts as ch

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
con = connect()
DIMS = {"industry": "Industry", "region_group": "Region (UK against EU)", "acquisition_channel": "Acquisition channel", "company_size": "Company size"}
SMALL = 30
cust = q(con, f"SELECT customer_id, {', '.join(DIMS)} FROM dim_customer")
rev = q(con, "SELECT customer_id, year(month_start) AS year, sum(net_revenue) AS net_revenue FROM fact_revenue_monthly GROUP BY ALL ORDER BY customer_id, year").merge(cust, on="customer_id")
dec = sql_file(con, "dec_mrr_customer_line").groupby(["customer_id", "year"], as_index=False).dec_mrr.sum().merge(cust, on="customer_id")

def seg_metrics(dim):
    r = rev.pivot_table(index=dim, columns="year", values="net_revenue", aggfunc="sum", fill_value=0)
    o = dec[dec.year == 2024].set_index("customer_id"); c = dec[dec.year == 2025].set_index("customer_id").dec_mrr
    o = o.assign(close=c.reindex(o.index, fill_value=0.0))
    g = o.groupby(dim).agg(customers_active_dec_2024=("dec_mrr", "size"), opening_mrr=("dec_mrr", "sum"), closing_mrr=("close", "sum"),
                            churned_2025=("close", lambda s: int((s == 0).sum())),
                            capped=("close", lambda s: 0.0))
    capped = o.assign(cap=np.minimum(o.close, o.dec_mrr)).groupby(dim).cap.sum()
    out = pd.DataFrame({"dimension": dim, "segment": r.index,
                        "net_revenue_2024": r[2024].values, "net_revenue_2025": r[2025].values}).set_index("segment")
    out["growth_2025_pct"] = 100 * (out.net_revenue_2025 / out.net_revenue_2024 - 1)
    out["share_of_2025_revenue_pct"] = 100 * out.net_revenue_2025 / out.net_revenue_2025.sum()
    out = out.join(g[["customers_active_dec_2024", "churned_2025", "opening_mrr", "closing_mrr"]])
    out["logo_churn_2025_pct"] = 100 * out.churned_2025 / out.customers_active_dec_2024
    out["nrr_2025_pct"] = 100 * out.closing_mrr / out.opening_mrr
    out["closing_mrr_capped"] = capped.reindex(out.index)
    out["grr_2025_pct"] = 100 * out.closing_mrr_capped / out.opening_mrr
    out["small_sample"] = out.customers_active_dec_2024 < SMALL
    return out.reset_index()

seg = pd.concat([seg_metrics(d) for d in DIMS], ignore_index=True)
o = dec[dec.year == 2024].set_index("customer_id"); c = dec[dec.year == 2025].set_index("customer_id").dec_mrr.reindex(o.index, fill_value=0.0)
company = {"logo_churn_2025_pct": 100 * (c == 0).mean(), "nrr_2025_pct": 100 * c.sum() / o.dec_mrr.sum(),
           "growth_2025_pct": 100 * (rev[rev.year == 2025].net_revenue.sum() / rev[rev.year == 2024].net_revenue.sum() - 1)}
seg["flag"] = np.select(
    [seg.small_sample,
     (seg.logo_churn_2025_pct > company["logo_churn_2025_pct"] + 5) | (seg.nrr_2025_pct < company["nrr_2025_pct"] - 5),
     (seg.logo_churn_2025_pct < company["logo_churn_2025_pct"] - 5) | (seg.nrr_2025_pct > company["nrr_2025_pct"] + 5)],
    ["too small to conclude", "risk", "strength"], default="")
seg = save_table(seg, "4g_segments")
save_table(pd.DataFrame([company]), "4g_company_benchmarks")
print(company)
seg[["dimension", "segment", "customers_active_dec_2024", "net_revenue_2025", "growth_2025_pct", "logo_churn_2025_pct", "nrr_2025_pct", "flag"]].round(1)

{'logo_churn_2025_pct': np.float64(12.732688011913625), 'nrr_2025_pct': np.float64(95.44347732071373), 'growth_2025_pct': np.float64(22.806688280705178)}


,dimension,segment,customers_active_dec_2024,net_revenue_2025,growth_2025_pct,logo_churn_2025_pct,nrr_2025_pct,flag
0,industry,Construction,146,4758042.6,21.7,15.1,97.2,
1,industry,Education,116,3860238.3,20.8,13.8,98.8,
2,industry,Financial Services,109,4686039.8,10.6,14.7,92.1,
3,industry,Healthcare,123,5434927.8,7.0,10.6,79.4,risk
4,industry,Hospitality,122,3876245.8,52.1,9.0,99.7,
5,industry,Logistics,122,3286180.3,32.3,9.8,97.6,
6,industry,Manufacturing,132,3744853.4,40.3,15.2,99.8,
7,industry,Professional Services,125,3160364.2,25.4,12.0,94.9,
8,industry,Retail,135,5833271.0,21.8,11.9,101.4,strength
9,industry,Technology,132,5010330.6,25.3,16.7,100.6,strength


## 2. Region detail

The EU is three countries. Each is shown so the sample sizes are visible.

In [2]:
cust_r = q(con, "SELECT customer_id, region FROM dim_customer")
rev_r = rev.merge(cust_r, on="customer_id"); dec_r = dec.merge(cust_r, on="customer_id")
rev, dec = rev_r, dec_r
region_detail = save_table(seg_metrics("region").assign(small_sample=lambda d: d.customers_active_dec_2024 < SMALL), "4g_region_detail")
region_detail[["segment", "customers_active_dec_2024", "net_revenue_2025", "growth_2025_pct", "logo_churn_2025_pct", "nrr_2025_pct", "small_sample"]].round(1)

,segment,customers_active_dec_2024,net_revenue_2025,growth_2025_pct,logo_churn_2025_pct,nrr_2025_pct,small_sample
0,Germany (EU),27,702070.3,15.3,11.1,99.7,True
1,Ireland (EU),35,754144.1,9.8,14.3,93.5,False
2,London,290,12412008.5,18.0,13.4,100.5,False
3,Midlands,157,3866934.1,23.8,17.8,91.5,False
4,Netherlands (EU),33,1933496.2,29.3,9.1,106.8,False
5,North East,73,2101416.6,29.5,12.3,99.3,False
6,North West,148,3795240.6,26.1,12.2,95.4,False
7,Northern Ireland,42,1476616.0,44.7,21.4,92.1,False
8,Scotland,89,2709042.8,44.0,7.9,100.7,False
9,South East,182,4712311.7,31.0,10.4,100.9,False


## 3. Tenure-matched view: 12-month logo retention by segment, older against newer cohorts

Share of customers still subscribed 12 months after signup, for customers who signed up in 2017 to 2022 against those who signed up from 2023 (only customers who had reached 12 months by December 2025).

In [3]:
act = sql_file(con, "cohort_logo_activity")
act["signup_month"] = pd.to_datetime(act.signup_month)
cell = act[(act.tenure == 12) & ((act.signup_month.dt.year * 12 + act.signup_month.dt.month + 12) <= 2025 * 12 + 12)]
cell = cell.merge(cust, on="customer_id")
cell["cohort_group"] = np.where(cell.signup_year >= 2023, "2023+ signups", "2017-2022 signups")
rows = []
for dim in DIMS:
    t = cell.groupby([dim, "cohort_group"]).active.agg(["mean", "size"]).unstack()
    for segname, r in t.iterrows():
        rows.append({"dimension": dim, "segment": segname,
                     "retention_12m_2017_2022_pct": 100 * r[("mean", "2017-2022 signups")], "customers_2017_2022": int(r[("size", "2017-2022 signups")]),
                     "retention_12m_2023_plus_pct": 100 * r[("mean", "2023+ signups")], "customers_2023_plus": int(r[("size", "2023+ signups")])})
tm = pd.DataFrame(rows)
tm["change_pts"] = tm.retention_12m_2023_plus_pct - tm.retention_12m_2017_2022_pct
tm["small_sample"] = tm.customers_2023_plus < SMALL
tm = save_table(tm, "4g_retention_12m_by_segment")
tm.round(1)

,dimension,segment,retention_12m_2017_2022_pct,customers_2017_2022,retention_12m_2023_plus_pct,customers_2023_plus,change_pts,small_sample
0,industry,Construction,91.7,133,78.8,66,-12.9,False
1,industry,Education,91.7,109,71.4,56,-20.3,False
2,industry,Financial Services,93.6,110,76.2,42,-17.4,False
3,industry,Healthcare,92.9,113,77.0,61,-15.9,False
4,industry,Hospitality,94.1,119,80.0,55,-14.1,False
5,industry,Logistics,89.7,116,86.0,57,-3.7,False
6,industry,Manufacturing,92.5,134,80.0,55,-12.5,False
7,industry,Professional Services,91.8,110,79.1,67,-12.7,False
8,industry,Retail,93.3,119,75.0,64,-18.3,False
9,industry,Technology,93.9,114,82.3,62,-11.6,False


## 4. Chart

In [4]:
fig, axes = plt.subplots(1, 4, figsize=(15, 5), sharex=True)
for ax, (dim, label) in zip(axes, DIMS.items()):
    d = tm[tm.dimension == dim].sort_values("retention_12m_2023_plus_pct")
    y = np.arange(len(d))
    ax.hlines(y, d.retention_12m_2023_plus_pct, d.retention_12m_2017_2022_pct, color=ch.GRID, lw=3)
    ax.scatter(d.retention_12m_2017_2022_pct, y, color=ch.BLUE, s=40, zorder=3, label="2017-2022 signups", edgecolor=ch.SURFACE)
    ax.scatter(d.retention_12m_2023_plus_pct, y, color=ch.ORANGE, s=40, zorder=3, label="2023+ signups", edgecolor=ch.SURFACE)
    ax.set_yticks(y); ax.set_yticklabels([f"{s}{' *' if sm else ''}" for s, sm in zip(d.segment, d.small_sample)], fontsize=7.5)
    ax.set_title(label, fontsize=10); ax.grid(axis="x"); ax.grid(axis="y", visible=False)
    ax.xaxis.set_major_formatter(ch.pct_fmt())
    ax.set_xlabel("Still subscribed after 12 months (%)")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper right", ncol=2, fontsize=8)
fig.suptitle("12-month logo retention by segment: older against newer cohorts (* fewer than 30 newer customers)", x=0.01, ha="left", fontsize=11, fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.94))
print(ch.save(fig, "4g_segment_retention"))

outputs/charts/4g_segment_retention.png


In [5]:
fig, ax = plt.subplots(figsize=(10, 5.2))
d = seg[~seg.small_sample].copy()
d["label"] = d.segment
# Four groups is one more than the palette's all-pairs safe set for scatter plots, so each group
# also has its own marker shape and every point is labelled: colour is never the only cue.
colours = {"industry": ch.BLUE, "region_group": ch.ORANGE, "acquisition_channel": ch.AQUA, "company_size": ch.YELLOW}
markers = {"industry": "o", "region_group": "s", "acquisition_channel": "^", "company_size": "D"}
for dim, g in d.groupby("dimension"):
    ax.scatter(g.logo_churn_2025_pct, g.nrr_2025_pct, s=np.clip(g.net_revenue_2025 / 4e4, 20, 400), color=colours[dim], alpha=0.85,
               marker=markers[dim], edgecolor=ch.SURFACE, linewidth=1.5, label=DIMS[dim])
    for _, r in g.iterrows():
        ax.annotate(r.label, (r.logo_churn_2025_pct, r.nrr_2025_pct), xytext=(4, 3), textcoords="offset points", fontsize=6.5, color=ch.TEXT_2)
ax.axvline(company["logo_churn_2025_pct"], color=ch.TEXT_2, lw=1, ls=(0, (3, 3)))
ax.axhline(company["nrr_2025_pct"], color=ch.TEXT_2, lw=1, ls=(0, (3, 3)))
ax.text(company["logo_churn_2025_pct"] + 0.2, ax.get_ylim()[1] - 1, f"company logo churn {company['logo_churn_2025_pct']:.1f}%", fontsize=7, color=ch.TEXT_2, va="top")
ax.xaxis.set_major_formatter(ch.pct_fmt()); ax.yaxis.set_major_formatter(ch.pct_fmt())
ax.set_xlabel("Logo churn 2025 (% of customers active in December 2024)"); ax.set_ylabel("NRR 2025 (%)")
ax.set_title("Segments: 2025 logo churn against NRR (bubble size: 2025 net revenue; small segments omitted)")
ax.legend(loc="lower left", fontsize=7.5, markerscale=0.5)
ax.grid(axis="x")
print(ch.save(fig, "4g_segments"))

outputs/charts/4g_segments.png
